# Developing chicken heart

This notebook runs the packaged `chicken_heart` workflow from data preparation
through downstream analysis. Edit the paths in **Setup**, then enable the run
switches for the steps you need. The saved outputs below come from the packaged
preset and do not require the external dataset.

## Setup

In [1]:
from pathlib import Path

import pandas as pd

from CytoBridge.workflow import (
    WorkflowOptions,
    build_workflow_plan,
    load_workflow_config,
    render_workflow_plan,
    run_workflow,
)
from CytoBridge.results import describe_figure_workflow

PRESET = 'chicken_heart'
RAW_H5AD = Path("data/chicken_heart_raw.h5ad")
OUTPUT_DIR = Path("tutorial_outputs/chicken_heart")
ALIGNED_H5AD = OUTPUT_DIR / "preprocess" / 'chicken_heart_aligned.h5ad'
MODEL_DIR = OUTPUT_DIR / "training"


import CytoBridge as cb

RAW_10X_DIR = Path("data/GSE149457_RAW")
METADATA_H5AD = Path("data/chicken_heart_spatial_merged_with_meta.h5ad")
REFERENCE_ALIGNMENT_H5AD = Path("data/heart_aligned_all_timepoints.h5ad")
PREPARATION_DIR = RAW_H5AD.parent / "chicken_heart_preparation"
RUN_RAW_DATA_ASSEMBLY = False


RUN_PREPARATION = False
RUN_PREPROCESS_AND_TRAIN = False
RUN_DOWNSTREAM = False

In [2]:
config, preset_source = load_workflow_config(PRESET)
dataset = config["dataset"]
scientific = config["scientific"]
downstream = config["downstream"]

pd.DataFrame(
    {
        "setting": [
            "dataset",
            "preset",
            "raw time column",
            "cell annotation",
            "observed training times",
            "classifier neighbors",
        ],
        "value": [
            dataset["display_name"],
            preset_source,
            config["preprocess"]["time_key"],
            dataset["annotation_key"],
            ", ".join(map(str, downstream["observed"])),
            scientific["classifier_k"],
        ],
    }
)

,setting,value
0,dataset,Developing chicken heart
1,preset,packaged preset: chicken_heart
2,raw time column,timepoint
3,cell annotation,celltype_prediction
4,observed training times,"0.0, 1.0, 2.0, 3.0"
5,classifier neighbors,1


## Data preparation

The preset records the count layer, time mapping, spatial coordinates, and
alignment settings used for this dataset. The plan below shows the input and
output paths before any long-running work starts.

### Assemble the chicken-heart H5AD

The downloaded 10x matrices are first matched to the reference spot roster.
The second call writes the `spatial_ot_input` coordinates expected by the
alignment preset.

In [3]:
if RUN_RAW_DATA_ASSEMBLY:
    PREPARATION_DIR.mkdir(parents=True, exist_ok=True)
    reference_input = PREPARATION_DIR / "chicken_heart_reference_input.h5ad"
    cb.pp.prepare_chicken_heart_input(
        raw_dir=RAW_10X_DIR,
        metadata_h5ad=METADATA_H5AD,
        aligned_reference_h5ad=REFERENCE_ALIGNMENT_H5AD,
        output_h5ad=reference_input,
        output_table=PREPARATION_DIR / "model_input.csv",
        manifest_path=PREPARATION_DIR / "preparation.json",
        graph_database=cb.pp.bundled_graph_database_path(PRESET),
        repair_legacy_d7_left_right=False,
    )
    cb.pp.prepare_chicken_heart_ot_input(
        input_h5ad=reference_input,
        output_h5ad=RAW_H5AD,
        output_table=PREPARATION_DIR / "chicken_heart_ot_input.csv",
        manifest_path=PREPARATION_DIR / "ot_input.json",
    )
else:
    print("Raw-data assembly is off. Set RUN_RAW_DATA_ASSEMBLY = True to run it.")

Raw-data assembly is off. Set RUN_RAW_DATA_ASSEMBLY = True to run it.


In [4]:
preparation_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess",),
)
preparation_plan = build_workflow_plan(
    config,
    source=preset_source,
    options=preparation_options,
)
print(render_workflow_plan(preparation_plan))

CytoBridge workflow plan
dataset: Developing chicken heart (chicken_heart)
config: packaged preset: chicken_heart
scientific parameters: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=1
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/chicken_heart/preprocess/chicken_heart_aligned.h5ad
    note: Use the raw-coordinate chicken-heart input with obsm['spatial_ot_input']. D7 is pre-oriented by a recorded 180-degree rotation around its raw-stage centroid, after which the package fits expression-guided OT alignment and a fresh edge predictor.
    edge predictor: not requested during preprocessing
  train: skipped; add --train to run (GPU required for production training)
  downstream: skipped (GPU recommended)


In [5]:
if RUN_PREPARATION:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before preprocessing: {RAW_H5AD}")
    preparation_result = run_workflow(config, options=preparation_options)
    preparation_result
else:
    print("Data preparation is off. Set RUN_PREPARATION = True to run it.")

Data preparation is off. Set RUN_PREPARATION = True to run it.


## Training

The full training run starts from the raw H5AD, writes the aligned data, fits
the interaction edge predictor when the preset requires one, and trains the
CytoBridge model. A production run requires a CUDA-capable environment.

In [6]:
training_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess", "train"),
    train=True,
)
training_plan = build_workflow_plan(
    config,
    source=preset_source,
    options=training_options,
)
print(render_workflow_plan(training_plan))

CytoBridge workflow plan
dataset: Developing chicken heart (chicken_heart)
config: packaged preset: chicken_heart
scientific parameters: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=1
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/chicken_heart/preprocess/chicken_heart_aligned.h5ad
    note: Use the raw-coordinate chicken-heart input with obsm['spatial_ot_input']. D7 is pre-oriented by a recorded 180-degree rotation around its raw-stage centroid, after which the package fits expression-guided OT alignment and a fresh edge predictor.
    edge predictor: will be trained automatically
      graph database: package: CytoBridge/workflow_databases/CellChatDB.ligrec.human.csv
      database source: bundled formal CellChatDB resource
      interaction cutoff: 0.21681429373719752
      decision threshold source: validation-selected during de novo training
      output: tutorial_outputs/chicken_heart/preprocess/edge_classifier/chic

In [7]:
if RUN_PREPROCESS_AND_TRAIN:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before training: {RAW_H5AD}")
    training_result = run_workflow(config, options=training_options)
    training_result
else:
    print("Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.")

Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.


## Downstream analysis

Downstream analysis uses the aligned H5AD and fitted model from the training
directory. The dataset preset supplies the interpolation times, classifier
settings, trajectory simulation, growth analysis, and ligand–receptor options.

In [8]:
downstream_options = WorkflowOptions(
    aligned_h5ad=ALIGNED_H5AD,
    model_dir=MODEL_DIR,
    output_dir=OUTPUT_DIR / "downstream",
    steps=("downstream",),
)
downstream_plan = build_workflow_plan(
    config,
    source=preset_source,
    options=downstream_options,
)
print(render_workflow_plan(downstream_plan))

CytoBridge workflow plan
dataset: Developing chicken heart (chicken_heart)
config: packaged preset: chicken_heart
scientific parameters: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=1
steps:
  preprocess: skipped (GPU for spatial alignment)
  train: skipped; add --train to run (GPU required for production training)
  downstream: ready (GPU recommended for SDE simulation and classifier fitting)
    model format: current
    output: tutorial_outputs/chicken_heart/downstream/downstream
    simulation: observed=[0.0, 1.0, 2.0, 3.0], interpolated=[0.5, 1.5, 2.5], initial particles=3550, dt=0.05, sigma=0.03, daughter noise=0, growth alpha=1, trajectory mode=piecewise_observed_anchored_interval_forward_simulation
      piecewise observed anchors: sample mode=per_timepoint, include end=False
      scope: Observed times use real cells. Each generated time is a one-sided, interval-local forward simulation initialized only from the immediately preceding observed anchor; it is not 

In [9]:
if RUN_DOWNSTREAM:
    missing = [path for path in (ALIGNED_H5AD, MODEL_DIR) if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing trained artifacts: {missing}")
    downstream_result = run_workflow(config, options=downstream_options)
    downstream_result
else:
    print("Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.")

Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.


## Paper figures

The paper notebooks below do not automatically consume `OUTPUT_DIR`. Each one
states whether it starts from a released compact result bundle, an external
figure release, or a complete upstream analysis. Check that route before using
the paper command with results from a new run.

- [Interaction-prior ablation](../paper_figures/lr_prior_ablation_stvcr.ipynb)
- [Five-dataset benchmark](../paper_figures/loto_benchmark.ipynb)
- [Training histories](../paper_figures/training_histories.ipynb)

In [10]:
figure_routes = [
    describe_figure_workflow(name)
    for name in ('interaction-evidence', 'loto-benchmark', 'training-histories')
]
pd.DataFrame(
    [
        {
            "paper location": route["paper_location"],
            "mode": route["mode"],
            "starts from": route["starts_from"],
            "command": route["figure_command"],
        }
        for route in figure_routes
    ]
)

,paper location,mode,starts from,command
0,Supplementary Figure S39,numeric-redraw,Packaged paired target-level error tables.,cytobridge figure interaction-evidence --outpu...
1,Supplementary Figure S40,numeric-redraw,Packaged target-level LOTO means and native-su...,cytobridge figure loto-benchmark --output-dir ...
2,Supplementary Figure S41,numeric-redraw,Packaged per-epoch histories and checkpoint su...,cytobridge figure training-histories --output-...


## Saved files

In [11]:
pd.DataFrame(
    {
        "file or directory": [
            "aligned data",
            "training directory",
            "downstream directory",
        ],
        "path": [
            str(ALIGNED_H5AD),
            str(MODEL_DIR),
            str(OUTPUT_DIR / "downstream"),
        ],
    }
)

,file or directory,path
0,aligned data,tutorial_outputs/chicken_heart/preprocess/chic...
1,training directory,tutorial_outputs/chicken_heart/training
2,downstream directory,tutorial_outputs/chicken_heart/downstream
